<a href="https://colab.research.google.com/github/DaminikaDz/Data_Ranking_Training_Dynamics/blob/main/src/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [44]:
!wget https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
!tar -xzf imagenette2-160.tgz

--2026-04-24 21:56:38--  https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.216.29.78, 52.216.38.40, 54.231.195.112, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.216.29.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 99003388 (94M) [application/x-tar]
Saving to: ‘imagenette2-160.tgz’

imagenette2-160.tgz 100%[===================>]  94.42M  45.7MB/s    in 2.1s    

2026-04-24 21:56:40 (45.7 MB/s) - ‘imagenette2-160.tgz’ saved [99003388/99003388]



In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [14]:
!pip install torch torchvision tqdm scikit-learn pandas pillow -q

In [8]:
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14")
model = model.to(device)
model.eval()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:10<00:00, 116MB/s]


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-23): 24 x NestedTensorBlock(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((1024,), eps=1e-06, element

In [9]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [45]:
train_ds = datasets.ImageFolder("imagenette2-160/train", transform=transform)
val_ds = datasets.ImageFolder("imagenette2-160/val", transform=transform)

print(len(train_ds), len(val_ds))

9469 3925


In [13]:
train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [14]:
def extract_embeddings(loader, model, device):
    all_embeddings = []
    all_labels = []
    all_paths = []

    with torch.inference_mode():
        for images, labels in tqdm(loader):
            images = images.to(device, non_blocking=True)

            emb = model(images)
            emb = F.normalize(emb, dim=1)

            all_embeddings.append(emb.cpu().numpy())
            all_labels.extend(labels.numpy())

    embeddings = np.vstack(all_embeddings)
    labels = np.array(all_labels)

    return embeddings, labels

In [15]:
train_embeddings, train_labels = extract_embeddings(train_loader, model, device)
val_embeddings, val_labels = extract_embeddings(val_loader, model, device)

print(train_embeddings.shape)
print(val_embeddings.shape)

100%|██████████| 62/62 [03:12<00:00,  3.10s/it]

(9469, 1024)
(3925, 1024)


In [17]:
!git clone https://github.com/facebookresearch/ssl-data-curation.git
%cd ssl-data-curation

Cloning into 'ssl-data-curation'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 35 (delta 3), reused 0 (delta 0), pack-reused 20 (from 1)
Receiving objects: 100% (35/35), 1.97 MiB | 5.96 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/ssl-data-curation


In [22]:
!pip install -r requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.6/811.6 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 96.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [18]:
from src.clusters import HierarchicalCluster
from src import (
    hierarchical_kmeans_gpu as hkmg,
    hierarchical_sampling as hs
)

In [19]:
data=torch.tensor(train_embeddings, device="cuda", dtype=torch.float32)

In [23]:
np.Inf = np.inf

In [33]:
clusters = hkmg.hierarchical_kmeans_with_resampling(
  data=torch.tensor(data, device="cuda", dtype=torch.float32),
  n_clusters=[1000, 300, 100],
  n_levels=3,
  sample_sizes=[5, 3, 2],
  verbose=True,
)

cl = HierarchicalCluster.from_dict(clusters)
sampled_indices = hs.hierarchical_sampling(cl, target_size=2000)

/tmp/ipykernel_10298/434840230.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  data=torch.tensor(data, device="cuda", dtype=torch.float32),


Hierarchical sampling from clusters: 100%|██████████| 100/100 [00:00<00:00, 2074.20it/s]


In [34]:
subset_labels = train_labels[sampled_indices]

pd.Series(subset_labels).value_counts().sort_index()

,count
0,192
1,177
2,199
3,199
4,333
5,188
6,176
7,173
8,175
9,188


In [36]:
from torch.utils.data import Subset
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

In [46]:
hkm_subset_ds = Subset(train_ds, sampled_indices)

In [38]:
batch_size = 64

full_train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
hkm_train_loader = DataLoader(hkm_subset_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

In [39]:
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = len(train_ds.classes)

def create_resnet50(frozen=True):
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights)

    if frozen:
        for p in model.parameters():
            p.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model.to(device)

In [40]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in tqdm(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [41]:
def run_experiment(name, train_loader, frozen=True, epochs=5):
    model = create_resnet50(frozen=frozen)
    criterion = nn.CrossEntropyLoss()

    if frozen:
        optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

    history = []

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        print(
            f"{name} | epoch {epoch+1}/{epochs} | "
            f"train_acc={train_acc:.4f} | val_acc={val_acc:.4f}"
        )

        history.append({
            "experiment": name,
            "epoch": epoch + 1,
            "frozen": frozen,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

    return pd.DataFrame(history)

In [47]:
results = []

results.append(run_experiment("full_frozen", full_train_loader, frozen=True, epochs=5))
results.append(run_experiment("hkm2000_frozen", hkm_train_loader, frozen=True, epochs=5))

results.append(run_experiment("full_unfrozen", full_train_loader, frozen=False, epochs=5))
results.append(run_experiment("hkm2000_unfrozen", hkm_train_loader, frozen=False, epochs=5))

results_df = pd.concat(results, ignore_index=True)

results_df

100%|██████████| 148/148 [00:36<00:00,  4.06it/s]


full_frozen | epoch 1/5 | train_acc=0.9333 | val_acc=0.9755


100%|██████████| 148/148 [00:33<00:00,  4.39it/s]


full_frozen | epoch 2/5 | train_acc=0.9799 | val_acc=0.9819


100%|██████████| 148/148 [00:34<00:00,  4.34it/s]


...

100%|██████████| 32/32 [00:21<00:00,  1.51it/s]


hkm2000_unfrozen | epoch 4/5 | train_acc=0.7340 | val_acc=0.7613


100%|██████████| 32/32 [00:21<00:00,  1.51it/s]


hkm2000_unfrozen | epoch 5/5 | train_acc=0.8330 | val_acc=0.8540


,experiment,epoch,frozen,train_loss,train_acc,val_loss,val_acc
0,full_frozen,1,True,0.511703,0.933256,0.157267,0.975541
1,full_frozen,2,True,0.110576,0.979935,0.091505,0.981911
2,full_frozen,3,True,0.067298,0.987433,0.071351,0.983949
3,full_frozen,4,True,0.048179,0.992185,0.065208,0.982930
4,full_frozen,5,True,0.036549,0.993030,0.056240,0.985732
5,hkm2000_frozen,1,True,1.409272,0.725500,0.914328,0.923312
6,hkm2000_frozen,2,True,0.560276,0.928000,0.484759,0.960255
7,hkm2000_frozen,3,True,0.343913,0.948500,0.342188,0.966369
8,hkm2000_frozen,4,True,0.249981,0.967500,0.266004,0.970701
9,hkm2000_frozen,5,True,0.196507,0.973500,0.218791,0.974013


In [48]:
new_result = run_experiment("hkm2000_unfrozen", hkm_train_loader, frozen=False, epochs=10)

100%|██████████| 32/32 [00:22<00:00,  1.39it/s]


hkm2000_unfrozen | epoch 1/10 | train_acc=0.1130 | val_acc=0.1890


100%|██████████| 32/32 [00:20<00:00,  1.55it/s]


hkm2000_unfrozen | epoch 2/10 | train_acc=0.3635 | val_acc=0.3896


100%|██████████| 32/32 [00:21<00:00,  1.52it/s]


...

hkm2000_unfrozen | epoch 8/10 | train_acc=0.9455 | val_acc=0.9429


100%|██████████| 32/32 [00:21<00:00,  1.50it/s]


hkm2000_unfrozen | epoch 9/10 | train_acc=0.9570 | val_acc=0.9503


100%|██████████| 32/32 [00:21<00:00,  1.51it/s]


hkm2000_unfrozen | epoch 10/10 | train_acc=0.9630 | val_acc=0.9513


In [51]:
new_result

,experiment,epoch,frozen,train_loss,train_acc,val_loss,val_acc
0,hkm2000_unfrozen,1,False,2.290371,0.1130,2.246793,0.189045
1,hkm2000_unfrozen,2,False,2.143022,0.3635,2.126994,0.389554
2,hkm2000_unfrozen,3,False,1.971374,0.5915,1.980823,0.570955
3,hkm2000_unfrozen,4,False,1.760891,0.7630,1.793502,0.744204
4,hkm2000_unfrozen,5,False,1.532789,0.8535,1.614839,0.857325
5,hkm2000_unfrozen,6,False,1.309027,0.9050,1.367134,0.911338
6,hkm2000_unfrozen,7,False,1.053981,0.9330,1.059143,0.931975
7,hkm2000_unfrozen,8,False,0.807287,0.9455,0.784008,0.942930
8,hkm2000_unfrozen,9,False,0.605654,0.9570,0.572767,0.950318
9,hkm2000_unfrozen,10,False,0.460984,0.9630,0.443734,0.951338
